In [ ]:


## Packages ---
import numpy as np
import pandas as pd
import getpass
from pathlib import Path
import os
import re
from tqdm import tqdm
from datetime import date
import time
import functools as ft
from IPython.display import display


## File paths ---

user = getpass.getuser()
path_users = Path.home()

path_sp = path_users / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents'
path_raw = path_sp / 'Process Revamp' / 'Task 9. Collect new data' / 'Census'
path_main = path_sp / 'Data'
path_prod = path_sp / 'Products'
path_dr = path_prod / 'Small Data Requests' / date.today().strftime('%Y')
path_git = path_users / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen'
path_code    = path_git / 'Data' / 'Census'
path_config0 = path_git / 'config'
path_config  = path_code / 'config'
path_server = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data")


## User defined functions ---

path_func = path_config0 / 'Functions.py'
path_func_census = path_config / 'census_functions.py'

with path_func.open("r") as f:
    exec(f.read())

with path_func_census.open("r") as f:
    exec(f.read())


In [ ]:

path_in  = path_server
estimate = 'ACS1'
geography = 'MSA'
workbook = f'Pop_9 {geography} {estimate}.xlsx'
sheet_name = geography


if geography == 'Places':
    geo_ID = ['State FIPS', 'County Name', 'Place ID', 'NAME']
if geography == 'Block Groups':
    geo_ID = ['State FIPS', 'County FIPS', 'County Name', 'Tract ID', 'Block Group ID', 'NAME']
if geography == 'Tracts':
    geo_ID = ['State FIPS', 'County FIPS', 'County Name', 'Tract ID', 'NAME']
if geography == 'Counties':
    geo_ID = ['State FIPS', 'MPO', 'County FIPS', 'County Name', 'NAME']
if geography == 'MPO':
    geo_ID = ['State FIPS', 'MPO']
if geography == 'MSA':
    geo_ID = ['MSA_ID', 'MSA']
if geography == 'Congressional Districts':
    geo_ID = ['State FIPS', 'Congressional District', 'NAME']
if geography == 'State Legislative Upper Districts':
    geo_ID = ['State FIPS', 'State Legislative Upper District', 'NAME']
if geography == 'State Legislative Lower Districts':
    geo_ID = ['State FIPS', 'State Legislative Lower District', 'NAME']
if geography == 'States':
    geo_ID = ['State FIPS', 'NAME']
if geography == 'National':
    geo_ID = ['NAME']


## Import ---


file_in = path_in / workbook
df_census = pd.read_excel(file_in, sheet_name=sheet_name)

df_census2 = df_census.copy()

df_census2 = df_census2.drop(['Race_Ethnicity', 'Margin of Error', 'Margin of Error Ratio', 'Use for Reporting'], axis=1)


# Reshape data to wide format
df_census2_pct = df_census2.copy()
df_census2_pct['Variable'] = df_census2_pct['Variable'] + '_pct'
df_census2_pct = df_census2_pct.pivot_table(index = geo_ID + ['Year']
                                    , columns = 'Variable'
                                    , values = 'Percentage').reset_index()
df_census2 = df_census2.pivot_table(index = geo_ID + ['Year']
                                    , columns = 'Variable'
                                    , values = 'Population').reset_index()
df_census2 = df_census2.sort_values(geo_ID + ['Year'], ascending = [item in geo_ID for item in geo_ID] + [False])
df_census2 = df_census2.merge(df_census2_pct, on= geo_ID+['Year'], how='left')

if geography == 'MSA':
    df_census2 = df_census2.sort_values(['MSA_ID', 'Year'], ascending = [True, False])

if geography == 'Counties':
    df_census2 = df_census2.sort_values(['MPO', 'Year'], ascending = [True, False])

display(df_census2)


## Exporting ---

df_about = pd.read_excel(file_in, sheet_name='About')
display(df_about)

workbook = f'Pop_9 {geography} {estimate}_v2.xlsx'
file_out = path_out / workbook

writer = pd.ExcelWriter(file_out, engine='xlsxwriter')

df_about  .to_excel(writer, sheet_name='About'  , index=False, header=False)
df_census2.to_excel(writer, sheet_name=geography, index=False, header=True )

writer.close()


In [ ]:
writer

**********************************************************************************************************************************

Plotting

**********************************************************************************************************************************

In [ ]:

export=False

indicator_name = 'Pop_9'
plot_name = f'Location 1 Year Ago_{geography}'
path_plots = path_out / 'Plots'
print('Export Location: ' + str(path_plots))

df_plot = df_census.copy()

display(df_plot.head())


if geography in ['MPO', 'States']:
    df_plot['Percentage'] = round(df_plot['Percentage'], 1)

    color_map = {
        'Same house':"#1F45FC"
        , 'Same county':"#9DC209"
        , 'Same city or town':'#1E90FF'
        , 'Elsewhere in CA':"#FBB117"
        , 'Elsewhere in U.S.':"#DC381F"
        , 'Abroad': '#7E587E'
    }

    fig = px.bar(df_plot, x='Year', y='Percentage', color='Variable', color_discrete_map=color_map)

    title = '<b>Location 1 Year Ago</b>  <br><sup>California</sup>'
    fig.update_yaxes(tick0=0, dtick=20, ticksuffix='%', range = [0, 102])
    fig.update_xaxes(tick0=0, dtick=1)
    fig.update_traces(hovertemplate='%{y}')
    fig.update_layout(legend={'traceorder': 'reversed'})

    plot_agol(export=export)



**********************************************************************************************************************************

Additional plots

**********************************************************************************************************************************

In [ ]:


export=False

indicator_name = 'Pop_9'
plot_name = f'Same House'
path_plots = path_out / 'Plots'
print('Export Location: ' + str(path_plots))


## Importing ---

geography = 'MPO'
wb_mpo = f"{indicator_name} {geography} {estimate}.xlsx"
file_mpo = path_server / wb_mpo
df_mpo = pd.read_excel(file_mpo, sheet_name = geography)

geography = 'MSA'
wb_msa = f"{indicator_name} {geography} {estimate}.xlsx"
file_msa = path_server / wb_msa
df_msa = pd.read_excel(file_msa, sheet_name = geography)

geography = 'States'
wb_cal = f"{indicator_name} {geography} {estimate}.xlsx"
file_cal = path_server / wb_cal
df_cal = pd.read_excel(file_cal, sheet_name = geography)



## Organizing ---

df_msa['Geography'] = 'Peer MSA'
df_mpo['Geography'] = 'SACOG'
df_cal['Geography'] = 'California'

df_msa = df_msa[~df_msa['Geography'].str.contains('Sac|Yuba')]

df_msa = df_msa.groupby(['Year', 'Geography', 'Race_Ethnicity', 'Variable'], as_index = False).agg(Population = ('Population', 'sum'))
df_msa['Percentage'] = 100*df_msa['Population'] / df_msa.groupby(['Geography', 'Year', 'Race_Ethnicity'])['Population'].transform('sum')

df_msa = df_msa[['Geography', 'Year', 'Variable', 'Percentage']]
df_mpo = df_mpo[['Geography', 'Year', 'Variable', 'Percentage']]
df_cal = df_cal[['Geography', 'Year', 'Variable', 'Percentage']]

df_msa = df_msa[(df_msa['Variable'] == 'Same house') & (df_msa['Year'] >= 2009)]
df_mpo = df_mpo[(df_mpo['Variable'] == 'Same house') & (df_mpo['Year'] >= 2009)]
df_cal = df_cal[(df_cal['Variable'] == 'Same house') & (df_cal['Year'] >= 2009)]

df_msa = df_msa.reset_index(drop=True)
df_mpo = df_mpo.reset_index(drop=True)
df_cal = df_cal.reset_index(drop=True)


df_plot = pd.concat([df_mpo, df_msa, df_cal])

df_plot = df_plot.reset_index(drop=True)
print(); print()
display(df_plot.head(), df_plot.tail())



df_plot['Percentage'] = round(df_plot['Percentage'], 1)

color_map = {
    'SACOG':"#1F45FC"
    , 'Peer MSA':"#9DC209"
    , 'California':'#1E90FF'
}

fig = px.line(df_plot, x='Year', y='Percentage', color='Geography', color_discrete_map=color_map, markers=True)

title = '<b>Location 1 Year Ago in the Same House (Percent of Population)</b>'
fig.update_yaxes(tick0=0, dtick=5, ticksuffix='%', range = [75, 92])
fig.update_xaxes(tick0=0, dtick=1)
fig.update_traces(hovertemplate='%{y}')
fig.update_layout(legend={'traceorder': 'reversed'})

plot_agol(export=export)

